# Final models

This notebook trains my 2 final models on all labelled steps 1-35 and saves their predictions for the test transactions in the `submissions` folder. The code and settings are the same as in `e1155604.ipynb` (Sections 4.1, 4.3, 7.2 and 7.3, cut-off 35), with all the other experiments removed.

1. **CC + spread of neighbour scores + SMOTE + PCA** (`final_cc_spread_smote_with_pca.csv`)
2. **CC + score PageRank + SMOTE + PCA** (`final_cc_pagerank_smote_with_pca.csv`)

The data files `train.csv`, `test.csv` and `txs_edgelist.csv` need to be in the `data` folder.

In [1]:
import numpy as np
import pandas as pd
import scipy.sparse as sp

import lightgbm as lgb

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import QuantileTransformer
from sklearn.decomposition import PCA
from imblearn.over_sampling import SMOTE

### 1. Load the data

In [2]:
train = pd.read_csv("data/train.csv")
test = pd.read_csv("data/test.csv")
edges = pd.read_csv("data/txs_edgelist.csv")

# 165 anonymised features
FEATS = [f"feat_{i}" for i in range(1, 166)]

print("Train shape:", train.shape)
print("Test shape:", test.shape)
print("Edges shape:", edges.shape)

Train shape: (141772, 168)
Test shape: (15329, 168)
Edges shape: (234355, 2)


### 2. Combine transactions and graph data

In [3]:
# combine train and test transactions
train_nodes = train.copy()
train_nodes["split"] = "train"

test_nodes = test.drop(columns="index").copy()
test_nodes["split"] = "test"
test_nodes["label"] = np.nan

nodes = pd.concat([train_nodes, test_nodes], ignore_index=True)

# transactions with available features
nodes["visible"] = (nodes["split"] == "test") | nodes["label"].notna()


# get all transaction IDs from the dataset and edge list
all_ids = np.concatenate([
    nodes["txId"].values,
    edges["txId1"].values,
    edges["txId2"].values
])

ids = pd.Index(pd.unique(all_ids))
N = len(ids)

# find the positions of the source and destination transactions
src = ids.get_indexer(edges["txId1"])
dst = ids.get_indexer(edges["txId2"])


# create adjacency matrices
A = sp.csr_matrix(
    (np.ones(len(src), dtype=np.float32), (src, dst)),
    shape=(N, N)
)
A.data[:] = 1

# reverse direction and undirected connections
AT = A.T.tocsr()
U = ((A + AT) > 0).astype(np.float32).tocsr()

# calculate the number of neighbours
degree = np.asarray(U.sum(axis=1)).ravel()

# averaging matrix over all neighbours, used for score PageRank
W = (sp.diags(1 / np.maximum(degree, 1)) @ U).tocsr()


# match transactions to their positions in the graph
gi = ids.get_indexer(nodes["txId"])

# indicate which graph nodes have available features
visible = np.zeros(N, dtype=np.float32)
visible[gi[nodes["visible"].values]] = 1

# store transaction features in graph order
X0 = np.zeros((N, len(FEATS)), dtype=np.float32)
X0[gi] = nodes[FEATS].values

print("Graph nodes:", N)
print("Graph edges:", len(src))

Graph nodes: 203769
Graph edges: 234355


### 3. Directional neighbour means

In [4]:
# calculate mean feature values of incoming and outgoing neighbours
neighbour_tables = []

for M, direction in [(AT, "vin"), (A, "vout")]:

    # only include neighbours with available features
    Mv = M.multiply(visible[None, :]).tocsr()

    # count the number of visible neighbours
    count = np.asarray(Mv.sum(axis=1)).ravel()

    # sum neighbour features and divide by the number of neighbours
    means = (Mv @ X0) / np.maximum(count, 1)[:, None]
    means = means.astype(np.float32)

    # set the mean to missing if there are no visible neighbours
    means[count == 0] = np.nan

    # give the features different names for incoming and outgoing neighbours
    columns = [f"{direction}_mean_{c}" for c in FEATS]

    neighbour_table = pd.DataFrame(means[gi], columns=columns, index=nodes.index)

    neighbour_tables.append(neighbour_table)


# add the neighbour features to the original transaction data
nodes = pd.concat([nodes] + neighbour_tables, axis=1)

# list of all directional neighbour features
DIR = [f"vin_mean_{c}" for c in FEATS]
DIR += [f"vout_mean_{c}" for c in FEATS]

# final feature set = raw features + directional neighbour means
SELECTED = FEATS + DIR

# labelled training and test transactions
L = nodes[(nodes["split"] == "train") & nodes["label"].notna()]
T = nodes[nodes["split"] == "test"]

print("Selected features:", len(SELECTED))

Selected features: 495


### 4. PCA fitted on steps 1-35

In [5]:
feature_rows = nodes[nodes["split"].isin(["train", "test"])]
fit_rows = feature_rows[feature_rows["time_step"] <= 35]

PCA_COLS = [f"pca_{i}" for i in range(10)]

# use quantile transformation first because many features are skewed
quantile = QuantileTransformer(output_distribution="normal", n_quantiles=1000, random_state=0)
scaled = quantile.fit_transform(fit_rows[FEATS])

# fit PCA on the training period, including unlabelled transactions
pca = PCA(n_components=10, random_state=0)
pca.fit(scaled)

# apply the same transformation to the other transactions
all_scaled = quantile.transform(feature_rows[FEATS])
components = pca.transform(all_scaled)

PCA_FEATURES = pd.DataFrame(components, columns=PCA_COLS, index=feature_rows.index)

print("Total variance explained:", round(pca.explained_variance_ratio_.sum(), 3))

Total variance explained: 0.705


### 5. First stage (give a score to every visible transaction)

In [6]:
# LightGBM settings used for all models
LGBM_PARAMS = dict(
    n_estimators=400,
    learning_rate=0.03,
    num_leaves=31,
    colsample_bytree=0.5,
    subsample=0.8,
    subsample_freq=1,
    verbose=-1,
    n_jobs=8
)

train_rows = L
other_rows = nodes[nodes["visible"] & ~nodes.index.isin(train_rows.index)]

with_pca_features = pd.concat([nodes[SELECTED], PCA_FEATURES], axis=1)
first = pd.Series(np.nan, index=nodes.index)

# out-of-fold scores for the training transactions, using 5 blocks of whole steps
blocks = np.array_split(np.array(sorted(train_rows["time_step"].unique())), 5)

for block in blocks:
    fit_part = train_rows[~train_rows["time_step"].isin(block)]
    held_out = train_rows[train_rows["time_step"].isin(block)]

    model = lgb.LGBMClassifier(**LGBM_PARAMS, random_state=0)
    model.fit(with_pca_features.loc[fit_part.index], fit_part["label"].astype(int))
    first[held_out.index] = model.predict_proba(with_pca_features.loc[held_out.index])[:, 1]

# scores for the test transactions, using all the training steps
model = lgb.LGBMClassifier(**LGBM_PARAMS, random_state=0)
model.fit(with_pca_features.loc[train_rows.index], train_rows["label"].astype(int))
first[other_rows.index] = model.predict_proba(with_pca_features.loc[other_rows.index])[:, 1]

### 6. Summaries of the neighbours' first-stage scores

In [7]:
# first-stage scores in graph order (unlabelled training transactions have no score)
P = np.zeros(N, np.float32)
has_score = np.zeros(N, np.float32)

ok = first.notna().values
P[gi[ok]] = first.values[ok]
has_score[gi[ok]] = 1

score_features = {}
spread_features = {}

for M, direction in [(AT, "in"), (A, "out"), (U, "all")]:
    # only neighbours that have a score count
    Mm = M.multiply(has_score[None, :]).tocsr()
    count = np.asarray(Mm.sum(axis=1)).ravel()

    mean = (Mm @ P) / np.maximum(count, 1)
    mx = np.asarray(Mm.multiply(P[None, :]).max(axis=1).todense()).ravel()
    spread = np.sqrt(np.maximum((Mm @ P ** 2) / np.maximum(count, 1) - mean ** 2, 0))

    mean[count == 0] = np.nan
    mx[count == 0] = np.nan
    spread[count == 0] = np.nan

    score_features[f"sp_{direction}_mean"] = mean[gi]
    score_features[f"sp_{direction}_max"] = mx[gi]
    spread_features[f"sp_{direction}_std"] = spread[gi]

# average score 2 hops away
scored = U.multiply(has_score[None, :]).tocsr()
R = sp.diags(1 / np.maximum(np.asarray(scored.sum(axis=1)).ravel(), 1)) @ scored
score_features["sp_2hop_mean"] = (R @ (R @ P))[gi]

# score PageRank, with the first-stage scores as the restart vector
num = P * has_score
den = has_score.copy()

for step in range(30):
    num = 0.85 * (W @ num) + 0.15 * P * has_score
    den = 0.85 * (W @ den) + 0.15 * has_score

extra = pd.DataFrame(spread_features, index=nodes.index)
extra["score_pagerank"] = (num / np.maximum(den, 1e-9))[gi]

basic = pd.DataFrame(score_features, index=nodes.index)

### 7. Second stage with SMOTE and saving the predictions

In [8]:
SPREAD_COLS = ["sp_in_std", "sp_out_std", "sp_all_std"]

# name -> (extra score features, file name)
FINALS = {
    "CC + spread of neighbour scores + PCA + SMOTE": (SPREAD_COLS, "final_cc_spread_smote_with_pca.csv"),
    "CC + score PageRank + PCA + SMOTE": (["score_pagerank"], "final_cc_pagerank_smote_with_pca.csv")
}

for name, (new_cols, file_name) in FINALS.items():
    features = pd.concat([with_pca_features, basic, extra[new_cols]], axis=1)
    X_train = features.loc[train_rows.index]
    X_target = features.loc[T.index]
    y = train_rows["label"].astype(int)

    # SMOTE can't handle missing values, so fill them with the median of the training rows first
    imputer = SimpleImputer(strategy="median")
    imputer.fit(X_train)

    X_train = pd.DataFrame(imputer.transform(X_train), columns=features.columns)
    X_target = pd.DataFrame(imputer.transform(X_target), columns=features.columns)

    # average over 3 seeds
    seed_scores = []

    for seed in [0, 1, 2]:
        X_fit, y_fit = SMOTE(random_state=seed).fit_resample(X_train, y.values)

        model = lgb.LGBMClassifier(**LGBM_PARAMS, random_state=seed)
        model.fit(X_fit, y_fit)
        seed_scores.append(model.predict_proba(X_target)[:, 1])

    submission = pd.DataFrame({"txId": T["txId"].values, "target": np.mean(seed_scores, axis=0)})

    # match the order and format of test.csv
    submission = test[["index", "txId"]].merge(submission, on="txId", how="left")[["index", "target"]]

    # check that every test transaction has a prediction
    assert len(submission) == len(test)
    assert submission["target"].notna().all()

    submission.to_csv(f"submissions/{file_name}", index=False)
    print("Saved", file_name, submission.shape)

Saved final_cc_spread_smote_with_pca.csv (15329, 2)
Saved final_cc_pagerank_smote_with_pca.csv (15329, 2)
